# RT-LCOD / DOD — Final End-to-End Notebook

**One notebook, one path:** install → official gRefCOCO + COCO images → prompt-independent YOLOE proposals → detector-independent RoI object tokens → frozen bidirectional text → A0 generic fusion + A1 explicit entity/relation reasoning → null/no-target training → calibration → held-out evaluation → `final_dod_bundle.pt` → reload and real inference.

Designed for **RTX A3000 12 GB**. Expensive image features are cached once per unique image and the detector/backbone are unloaded before training. Set `cfg.quick=False` for the final run; `quick=True` is only a pipeline smoke run.

In [ ]:
%pip install -q -U "ultralytics>=8.4.0" "transformers>=5.1,<6" "huggingface-hub>=0.35" "pycocotools>=2.0.7" "opencv-python>=4.10" "scikit-learn>=1.5" "matplotlib>=3.9" "pandas>=2.2" "tqdm>=4.66" "requests>=2.32" "Pillow>=10"


In [ ]:
from __future__ import annotations
import gc, re, json, math, random, shutil, hashlib, concurrent.futures
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, requests
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import roi_align, nms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download

@dataclass
class Config:
    seed:int=1337
    root:str='../data/dod_final'
    artifact:str='../artifacts/dod_final_model'
    quick:bool=False
    quick_train:int=3000
    quick_val:int=600
    quick_test:int=600
    dataset_repo:str='FudanCVL/gRefCOCO'
    download_mode:str='needed'  # needed | zip
    download_workers:int=16
    proposal_model:str='yoloe-26s-seg-pf.pt'
    proposal_imgsz:int=640
    proposal_conf:float=0.02
    proposal_max_det:int=96
    top_k:int=64
    region_size:int=512
    roi_size:int=3
    match_iou:float=0.50
    gt_injection:bool=True
    text_model:str='distilbert-base-uncased'
    max_text_len:int=48
    d_model:int=256
    heads:int=8
    object_layers:int=2
    cross_layers:int=2
    ff_dim:int=512
    dropout:float=0.10
    anchor_top_m:int=8
    epochs:int=10
    batch_size:int=16
    workers:int=0  # reliable default for Windows/Jupyter
    lr:float=3e-4
    weight_decay:float=1e-4
    grad_clip:float=1.0
    patience:int=3
    w_candidate:float=1.0
    w_null:float=0.45
    w_entity:float=0.20
    nms_iou:float=0.50
    max_outputs:int=20
    export_yoloe_tensorrt:bool=False

cfg=Config()
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.seed)
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not cfg.quick and DEV.type!='cuda':
    raise RuntimeError('Full run requires CUDA PyTorch. Install a CUDA-enabled torch build, restart kernel, then Run All.')
print('torch',torch.__version__,'device',DEV)
if DEV.type=='cuda':
    print('GPU',torch.cuda.get_device_name(0),'VRAM GB',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
ROOT=Path(cfg.root).resolve(); ART=Path(cfg.artifact).resolve(); ANN=ROOT/'annotations'; IMG=ROOT/'train2014'; CK=ROOT/'checkpoints'
for p in [ROOT,ART,ANN,IMG,CK]: p.mkdir(parents=True,exist_ok=True)


## 1. Official data → canonical 0/1/N DOD rows

The notebook uses official `FudanCVL/gRefCOCO` annotations and MS-COCO train2014 images. Each referring sentence becomes one sample, while image-side computation is cached once per unique image.

In [ ]:
for name in ['grefs(unc).json','instances.json']:
    dst=ANN/name
    if not dst.exists():
        src=hf_hub_download(cfg.dataset_repo,name,repo_type='dataset')
        shutil.copy2(src,dst)
refs=json.load(open(ANN/'grefs(unc).json',encoding='utf-8'))
inst=json.load(open(ANN/'instances.json',encoding='utf-8'))
anns={int(a['id']):a for a in inst['annotations']}
imgs={int(i['id']):i for i in inst['images']}
cats={int(c['id']):c['name'] for c in inst['categories']}

def as_list(x): return x if isinstance(x,list) else ([] if x is None else [x])
def xywh_to_xyxy(b):
    x,y,w,h=map(float,b); return [x,y,x+w,y+h]

rows=[]; image_ann_ids=defaultdict(set)
for r in refs:
    image_id=int(r['image_id'])
    ann_ids=[] if r.get('no_target',False) else [int(x) for x in as_list(r.get('ann_id')) if int(x)>=0 and int(x) in anns]
    for a in ann_ids: image_ann_ids[image_id].add(a)
    boxes=[xywh_to_xyxy(anns[a]['bbox']) for a in ann_ids]
    category_ids=[int(x) for x in as_list(r.get('category_id')) if int(x) in cats]
    file_name=r.get('file_name') or imgs[image_id]['file_name']
    for s in r.get('sentences',[]):
        q=(s.get('sent') or s.get('raw') or '').strip()
        if q:
            rows.append(dict(sample_id=f"{r['ref_id']}:{s.get('sent_id',-1)}",image_id=image_id,file_name=file_name,split=str(r['split']),query=q,gt_boxes=boxes,no_target=len(boxes)==0,categories=[cats[x] for x in category_ids]))
df=pd.DataFrame(rows)
train_df=df[df.split.eq('train')].reset_index(drop=True)
val_df=df[df.split.eq('val')].reset_index(drop=True)
test_df=df[df.split.isin(['test','testA','testB'])].reset_index(drop=True)
if len(test_df)==0:
    raise RuntimeError('No held-out gRefCOCO test rows were found; inspect split names before training.')
if cfg.quick:
    train_df=train_df.sample(min(cfg.quick_train,len(train_df)),random_state=1).reset_index(drop=True)
    val_df=val_df.sample(min(cfg.quick_val,len(val_df)),random_state=2).reset_index(drop=True)
    test_df=test_df.sample(min(cfg.quick_test,len(test_df)),random_state=3).reset_index(drop=True)
active=pd.concat([train_df,val_df,test_df],ignore_index=True)
print({'train':len(train_df),'val':len(val_df),'test':len(test_df),'unique_images':active.image_id.nunique(),'no_target_rate':float(active.no_target.mean())})
assert len(train_df)>0 and len(val_df)>0


In [ ]:
# Download only images needed by the active run by default. This is much lighter than a full train2014 zip.
needed=sorted(set(active.file_name.tolist()))
def fetch_one(name):
    dst=IMG/name
    if dst.exists() and dst.stat().st_size>1000: return True
    url=f'https://images.cocodataset.org/train2014/{name}'
    for attempt in range(3):
        try:
            r=requests.get(url,timeout=60); r.raise_for_status(); dst.write_bytes(r.content)
            return dst.stat().st_size>1000
        except Exception:
            if attempt==2: return False
    return False
if cfg.download_mode=='needed':
    missing=[x for x in needed if not (IMG/x).exists()]
    if missing:
        with concurrent.futures.ThreadPoolExecutor(max_workers=cfg.download_workers) as ex:
            ok=list(tqdm(ex.map(fetch_one,missing),total=len(missing),desc='COCO images'))
        if not all(ok): raise RuntimeError(f'{sum(not x for x in ok)} COCO images failed to download')
else:
    z=ROOT/'train2014.zip'
    if not z.exists():
        with requests.get('https://images.cocodataset.org/zips/train2014.zip',stream=True,timeout=120) as r:
            r.raise_for_status()
            with open(z,'wb') as f:
                for ch in tqdm(r.iter_content(1024*1024),desc='COCO zip'):
                    if ch: f.write(ch)
    import zipfile
    with zipfile.ZipFile(z) as zz: zz.extractall(ROOT)
missing=[x for x in needed if not (IMG/x).exists()]
if missing: raise FileNotFoundError(f'Missing {len(missing)} active images; first={missing[:3]}')
print('images ready',len(needed))


## 2. QueryGraph and geometry

A0 always consumes the original full sentence. A1 additionally uses an explicit target/anchor/relation parse. If no supported relation is parsed, A1 reduces to target-entity matching and A0 remains the unrestricted semantic path.

In [ ]:
REL_PATTERNS=[(r'\bto the left of\b|\bleft of\b','left_of'),(r'\bto the right of\b|\bright of\b','right_of'),(r'\bin front of\b','in_front_of'),(r'\bbehind\b','behind'),(r'\babove\b|\bover\b','above'),(r'\bbelow\b|\bunder\b','below'),(r'\bnext to\b|\bbeside\b|\bnear\b','near'),(r'\binside\b','inside'),(r'\bon top of\b','on_top_of')]
REL2ID={'none':0,**{name:i+1 for i,(_,name) in enumerate(REL_PATTERNS)}}

def parse_query(q):
    low=q.lower().strip()
    for pat,rel in REL_PATTERNS:
        m=re.search(pat,low)
        if m:
            t=low[:m.start()].strip(' ,') or 'object'; a=low[m.end():].strip(' ,') or 'object'
            return t,a,rel,True
    return low or 'object','object','none',False

def box_iou(a,b):
    if a.numel()==0 or b.numel()==0: return a.new_zeros((a.shape[0],b.shape[0]))
    lt=torch.maximum(a[:,None,:2],b[None,:,:2]); rb=torch.minimum(a[:,None,2:],b[None,:,2:]); wh=(rb-lt).clamp_min(0)
    inter=wh[...,0]*wh[...,1]; aa=((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; bb=((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return inter/(aa+bb-inter).clamp_min(1e-6)

def box_features(b):
    x1,y1,x2,y2=b.unbind(-1); w=(x2-x1).clamp_min(1e-6); h=(y2-y1).clamp_min(1e-6)
    return torch.stack([(x1+x2)/2,(y1+y2)/2,w,h,w*h,torch.log(w/h)],-1)

def pair_geometry(a,b):
    ax1,ay1,ax2,ay2=a.unbind(-1); bx1,by1,bx2,by2=b.unbind(-1)
    aw=(ax2-ax1).clamp_min(1e-6); ah=(ay2-ay1).clamp_min(1e-6); bw=(bx2-bx1).clamp_min(1e-6); bh=(by2-by1).clamp_min(1e-6)
    acx=(ax1+ax2)/2; acy=(ay1+ay2)/2; bcx=(bx1+bx2)/2; bcy=(by1+by2)/2
    dx=(acx-bcx)/bw; dy=(acy-bcy)/bh
    ix1=torch.maximum(ax1,bx1); iy1=torch.maximum(ay1,by1); ix2=torch.minimum(ax2,bx2); iy2=torch.minimum(ay2,by2)
    inter=(ix2-ix1).clamp_min(0)*(iy2-iy1).clamp_min(0); union=aw*ah+bw*bh-inter
    return torch.stack([dx,dy,torch.sqrt(dx*dx+dy*dy+1e-8),torch.log(aw/bw),torch.log(ah/bh),bx1-ax2,ax1-bx2,by1-ay2,ay1-by2,inter/union.clamp_min(1e-6)],-1)


## 3. Prompt-independent proposals + detector-independent RoI object features

The cache is keyed by a configuration signature. Stale caches from a different K/resolution/model are therefore not reused. GT RoI features are cached only to support training-only missed-GT injection; validation/test never inject GT.

In [ ]:
from ultralytics import YOLOE

class RegionEncoder(nn.Module):
    output_dim=576
    def __init__(self,pretrained=True):
        super().__init__()
        w=MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        self.backbone=mobilenet_v3_small(weights=w).features
        self.register_buffer('mean',torch.tensor([.485,.456,.406]).view(1,3,1,1)); self.register_buffer('std',torch.tensor([.229,.224,.225]).view(1,3,1,1))
        for p in self.parameters(): p.requires_grad_(False)
        self.eval()
    @torch.inference_mode()
    def encode(self,pil,boxes):
        if len(boxes)==0: return torch.zeros((0,self.output_dim),dtype=torch.float16)
        device=self.mean.device; arr=np.asarray(pil.convert('RGB'),dtype=np.float32)/255.; h,w=arr.shape[:2]
        x=torch.from_numpy(arr.copy()).permute(2,0,1)[None].to(device); x=F.interpolate(x,(cfg.region_size,cfg.region_size),mode='bilinear',align_corners=False)
        fm=self.backbone((x-self.mean)/self.std)
        b=boxes.to(device).float().clone(); b[:,[0,2]]*=cfg.region_size/w; b[:,[1,3]]*=cfg.region_size/h
        rois=torch.cat([torch.zeros((len(b),1),device=device),b],1)
        return roi_align(fm,rois,(cfg.roi_size,cfg.roi_size),spatial_scale=1/32,aligned=True).mean((-1,-2)).cpu().half()

cache_signature=hashlib.sha256(json.dumps({'proposal':cfg.proposal_model,'imgsz':cfg.proposal_imgsz,'conf':cfg.proposal_conf,'max_det':cfg.proposal_max_det,'K':cfg.top_k,'region':cfg.region_size,'roi':cfg.roi_size},sort_keys=True).encode()).hexdigest()[:12]
CACHE=ROOT/'candidate_cache'/cache_signature; CACHE.mkdir(parents=True,exist_ok=True)
json.dump({'signature':cache_signature,'config':asdict(cfg)},open(CACHE/'cache_meta.json','w'),indent=2)
proposal=YOLOE(cfg.proposal_model); region=RegionEncoder(True).to(DEV)
torch.save(region.state_dict(),ROOT/'region_encoder.pt')

# Union of all annotated boxes for each active image; used only to cache visual features for possible train-time injection.
active_gt=defaultdict(list)
for r in active.itertuples():
    for b in r.gt_boxes: active_gt[int(r.image_id)].append(b)
for k,v in active_gt.items():
    uniq=[]
    for b in v:
        if tuple(round(float(x),2) for x in b) not in {tuple(round(float(x),2) for x in z) for z in uniq}: uniq.append(b)
    active_gt[k]=uniq

unique_images=active[['image_id','file_name']].drop_duplicates().itertuples(index=False)
for row in tqdm(list(unique_images),desc='cache image proposals/features'):
    out=CACHE/f'{int(row.image_id)}.pt'
    if out.exists(): continue
    pil=Image.open(IMG/row.file_name).convert('RGB'); w,h=pil.size
    res=proposal.predict(source=np.asarray(pil),imgsz=cfg.proposal_imgsz,conf=cfg.proposal_conf,max_det=cfg.proposal_max_det,device=0 if DEV.type=='cuda' else 'cpu',half=DEV.type=='cuda',verbose=False)[0]
    if res.boxes is None or len(res.boxes)==0:
        boxes=torch.zeros((0,4)); scores=torch.zeros(0); labels=[]
    else:
        boxes=res.boxes.xyxy.detach().float().cpu(); scores=res.boxes.conf.detach().float().cpu(); clsids=res.boxes.cls.detach().long().cpu()
        order=torch.argsort(scores,descending=True)[:cfg.top_k]; boxes=boxes[order]; scores=scores[order]; labels=[str(res.names[int(clsids[i])]) for i in order.tolist()]
    feat=region.encode(pil,boxes)
    gb=torch.tensor(active_gt.get(int(row.image_id),[]),dtype=torch.float32).reshape(-1,4); gf=region.encode(pil,gb)
    torch.save({'image_id':int(row.image_id),'file_name':row.file_name,'width':w,'height':h,'boxes':boxes,'scores':scores,'labels':labels,'features':feat,'gt_bank_boxes':gb,'gt_bank_features':gf},out)

# Copy locally resolved proposal checkpoint when possible. Runtime can redownload by model id if not found.
for candidate in [Path(cfg.proposal_model),Path.cwd()/cfg.proposal_model,Path(str(getattr(proposal,'ckpt_path','')) if getattr(proposal,'ckpt_path',None) else '__none__')]:
    if candidate.exists() and candidate.is_file(): shutil.copy2(candidate,ROOT/'proposal_model.pt'); break
del proposal,region; gc.collect();
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('cache',CACHE,'files',len(list(CACHE.glob('*.pt'))))


## 4. Frozen text encoder and semantic-prior vocabulary

DistilBERT is frozen. Full-query token states drive A0; pooled target/anchor phrases drive A1. Detector labels are encoded once and used only as an optional prior inside object tokens.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(cfg.text_model)
text_encoder=AutoModel.from_pretrained(cfg.text_model).to(DEV).eval()
for p in text_encoder.parameters(): p.requires_grad_(False)
TEXT_DIM=int(text_encoder.config.hidden_size)
@torch.inference_mode()
def encode_text(texts,max_len=None):
    toks=tokenizer(list(texts),padding=True,truncation=True,max_length=max_len or cfg.max_text_len,return_tensors='pt'); toks={k:v.to(DEV) for k,v in toks.items()}
    z=text_encoder(**toks).last_hidden_state; mask=toks['attention_mask'].bool(); pooled=(z*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
    return z,mask,pooled
label_vocab=sorted({label for p in CACHE.glob('*.pt') if p.name!='cache_meta.json' for label in torch.load(p,map_location='cpu',weights_only=False)['labels']})
LABEL2ID={x:i+1 for i,x in enumerate(label_vocab)}  # 0 = missing/injected GT
if label_vocab:
    blocks=[]
    for i in tqdm(range(0,len(label_vocab),128),desc='encode detector labels'):
        _,_,v=encode_text(label_vocab[i:i+128],20); blocks.append(v.cpu())
    LABEL_EMB=torch.cat([torch.zeros((1,TEXT_DIM)),torch.cat(blocks,0)],0)
else: LABEL_EMB=torch.zeros((1,TEXT_DIM))
print('text dim',TEXT_DIM,'encountered detector labels',len(label_vocab))


## 5. Dataset, train-only GT injection, and labels

Each GT claims at most one candidate (greedy IoU assignment). If a training GT has no proposal at IoU≥0.5, its cached RoI feature replaces a low-score unprotected proposal; it receives detector score 0 and no detector-label prior. This never happens in validation/test.

In [ ]:
def assign_targets(boxes,gt,thr):
    y=torch.zeros(len(boxes)); used=set()
    if len(boxes)==0 or len(gt)==0: return y
    M=box_iou(boxes,gt)
    for j in range(len(gt)):
        for i in torch.argsort(M[:,j],descending=True).tolist():
            if float(M[i,j])<thr: break
            if i not in used: y[i]=1.; used.add(i); break
    return y

def prepare_candidates(cache,gt_boxes,train_mode):
    boxes=cache['boxes'].float().clone(); scores=cache['scores'].float().clone(); features=cache['features'].float().clone(); labels=list(cache['labels'])
    gt=torch.tensor(gt_boxes,dtype=torch.float32).reshape(-1,4)
    injected=0
    if train_mode and cfg.gt_injection and len(gt):
        miss=[]
        M=box_iou(boxes,gt) if len(boxes) else torch.zeros((0,len(gt)))
        for j in range(len(gt)):
            if len(boxes)==0 or float(M[:,j].max())<cfg.match_iou: miss.append(j)
        if miss:
            protected=set(torch.where((M.max(1).values>=cfg.match_iou) if len(boxes) else torch.zeros(0,dtype=torch.bool))[0].tolist())
            bank=cache['gt_bank_boxes'].float(); bankf=cache['gt_bank_features'].float()
            for j in miss:
                if len(bank)==0: continue
                bi=int(box_iou(gt[j:j+1],bank)[0].argmax()); newb=gt[j].clone(); newf=bankf[bi].clone()
                if len(boxes)<cfg.top_k:
                    boxes=torch.cat([boxes,newb[None]]); scores=torch.cat([scores,torch.zeros(1)]); features=torch.cat([features,newf[None]]); labels.append('')
                else:
                    choices=[i for i in torch.argsort(scores).tolist() if i not in protected]
                    if not choices: continue
                    k=choices[0]; boxes[k]=newb; scores[k]=0.; features[k]=newf; labels[k]=''; protected.add(k)
                injected+=1
    order=torch.argsort(scores,descending=True)[:cfg.top_k] if len(scores) else torch.zeros(0,dtype=torch.long)
    boxes=boxes[order]; scores=scores[order]; features=features[order]; labels=[labels[i] for i in order.tolist()]
    y=assign_targets(boxes,gt,cfg.match_iou)
    return boxes,scores,features,labels,y,gt,injected

class DODDataset(Dataset):
    def __init__(self,frame,train_mode=False): self.frame=frame.reset_index(drop=True); self.train_mode=train_mode
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]; c=torch.load(CACHE/f"{int(r.image_id)}.pt",map_location='cpu',weights_only=False)
        boxes,scores,features,labels,y,gt,inj=prepare_candidates(c,r.gt_boxes,self.train_mode)
        scale=torch.tensor([c['width'],c['height'],c['width'],c['height']],dtype=torch.float32)
        return {'sample_id':r.sample_id,'image_id':int(r.image_id),'file_name':r.file_name,'query':r.query,'boxes':boxes/scale,'scores':scores,'features':features,'label_ids':torch.tensor([LABEL2ID.get(x,0) for x in labels]),'y':y,'gt':gt/scale,'null':float(len(gt)==0),'injected':inj}

def collate(batch):
    B=len(batch); K=cfg.top_k; D=576
    boxes=torch.zeros((B,K,4)); scores=torch.zeros((B,K)); features=torch.zeros((B,K,D)); label_ids=torch.zeros((B,K),dtype=torch.long); mask=torch.zeros((B,K),dtype=torch.bool); y=torch.zeros((B,K)); null=torch.tensor([x['null'] for x in batch],dtype=torch.float32)
    queries=[]; targets=[]; anchors=[]; rel=[]; has_rel=[]; meta=[]
    for n,x in enumerate(batch):
        k=min(K,len(x['boxes'])); boxes[n,:k]=x['boxes'][:k]; scores[n,:k]=x['scores'][:k]; features[n,:k]=x['features'][:k]; label_ids[n,:k]=x['label_ids'][:k]; mask[n,:k]=True; y[n,:k]=x['y'][:k]
        t,a,r,h=parse_query(x['query']); queries.append(x['query']); targets.append(t); anchors.append(a); rel.append(REL2ID[r]); has_rel.append(h); meta.append({'sample_id':x['sample_id'],'image_id':x['image_id'],'file_name':x['file_name'],'gt':x['gt'],'injected':x['injected']})
    return {'boxes':boxes,'scores':scores,'features':features,'label_ids':label_ids,'mask':mask,'y':y,'null':null,'queries':queries,'targets':targets,'anchors':anchors,'rel':torch.tensor(rel),'has_rel':torch.tensor(has_rel,dtype=torch.bool),'meta':meta}

train_loader=DataLoader(DODDataset(train_df,True),batch_size=cfg.batch_size,shuffle=True,num_workers=cfg.workers,pin_memory=DEV.type=='cuda',collate_fn=collate)
val_loader=DataLoader(DODDataset(val_df,False),batch_size=cfg.batch_size,shuffle=False,num_workers=cfg.workers,pin_memory=DEV.type=='cuda',collate_fn=collate)
test_loader=DataLoader(DODDataset(test_df,False),batch_size=cfg.batch_size,shuffle=False,num_workers=cfg.workers,pin_memory=DEV.type=='cuda',collate_fn=collate)


## 6. Final A0+A1 architecture

**A0**: self-attention among object tokens, then candidate→full-text cross-attention. **A1**: explicit target/anchor entity matching plus relation-conditioned pair geometry over top-M anchors. A learned gate adds A1 structured evidence to A0. An independent null head learns absent-target rejection.

In [ ]:
class CrossBlock(nn.Module):
    def __init__(self,d,h,ff,drop):
        super().__init__(); self.attn=nn.MultiheadAttention(d,h,dropout=drop,batch_first=True); self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d); self.ff=nn.Sequential(nn.Linear(d,ff),nn.GELU(),nn.Dropout(drop),nn.Linear(ff,d),nn.Dropout(drop))
    def forward(self,x,t,text_mask):
        a,_=self.attn(x,t,t,key_padding_mask=~text_mask,need_weights=False); x=self.n1(x+a); return self.n2(x+self.ff(x))

class FinalDODCore(nn.Module):
    def __init__(self,visual_dim=576,text_dim=768):
        super().__init__(); d=cfg.d_model
        self.visual=nn.Linear(visual_dim,d); self.box=nn.Sequential(nn.Linear(6,d),nn.GELU(),nn.Linear(d,d)); self.score=nn.Linear(1,d); self.label=nn.Linear(text_dim,d); self.obj_norm=nn.LayerNorm(d)
        enc=nn.TransformerEncoderLayer(d,cfg.heads,cfg.ff_dim,cfg.dropout,batch_first=True,norm_first=True,activation='gelu'); self.object_encoder=nn.TransformerEncoder(enc,cfg.object_layers)
        self.text_proj=nn.Linear(text_dim,d); self.cross=nn.ModuleList([CrossBlock(d,cfg.heads,cfg.ff_dim,cfg.dropout) for _ in range(cfg.cross_layers)]); self.generic=nn.Sequential(nn.Linear(d,d),nn.GELU(),nn.Linear(d,1))
        self.entity_obj=nn.Linear(d,d); self.entity_text=nn.Linear(text_dim,d); self.rel_emb=nn.Embedding(len(REL2ID),d); self.geom=nn.Sequential(nn.Linear(10,64),nn.GELU(),nn.Linear(64,64)); self.rel_mlp=nn.Sequential(nn.Linear(d*3+64,d),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(d,1))
        self.gate=nn.Sequential(nn.Linear(text_dim+1,d//2),nn.GELU(),nn.Linear(d//2,1)); self.null=nn.Sequential(nn.Linear(d+text_dim+1,d),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(d,1))
    def forward(self,visual,boxes,scores,label_emb,mask,text_tokens,text_mask,query_pool,target_pool,anchor_pool,relation_id,has_relation):
        x=self.obj_norm(self.visual(visual)+self.box(box_features(boxes))+self.score(scores[...,None])+self.label(label_emb)); x=self.object_encoder(x,src_key_padding_mask=~mask)
        t=self.text_proj(text_tokens)
        for block in self.cross: x=block(x,t,text_mask)
        generic=self.generic(x).squeeze(-1)
        oe=F.normalize(self.entity_obj(x),dim=-1); tq=F.normalize(self.entity_text(target_pool),dim=-1); aq=F.normalize(self.entity_text(anchor_pool),dim=-1)
        target_score=4*(oe*tq[:,None]).sum(-1); anchor_score=4*(oe*aq[:,None]).sum(-1); target_score=target_score.masked_fill(~mask,-1e4); anchor_score=anchor_score.masked_fill(~mask,-1e4)
        M=min(cfg.anchor_top_m,x.shape[1]); anchor_idx=anchor_score.topk(M,dim=1).indices; bi=torch.arange(x.shape[0],device=x.device)[:,None]; ax=x[bi,anchor_idx]; ab=boxes[bi,anchor_idx]; asc=anchor_score[bi,anchor_idx]
        xi=x[:,:,None,:].expand(-1,-1,M,-1); xj=ax[:,None,:,:].expand(-1,x.shape[1],-1,-1); aa=boxes[:,:,None,:].expand(-1,-1,M,-1); bb=ab[:,None,:,:].expand(-1,x.shape[1],-1,-1); gg=self.geom(pair_geometry(aa,bb)); rr=self.rel_emb(relation_id)[:,None,None,:].expand(-1,x.shape[1],M,-1)
        pair=self.rel_mlp(torch.cat([xi,xj,rr,gg],-1)).squeeze(-1)+asc[:,None,:]
        self_pair=torch.arange(x.shape[1],device=x.device)[None,:,None].eq(anchor_idx[:,None,:]); pair=pair.masked_fill(self_pair,-1e4); rel_score=torch.logsumexp(pair,dim=-1)-math.log(max(M,1))
        structured=target_score+torch.where(has_relation[:,None],rel_score,torch.zeros_like(rel_score)); alpha=torch.sigmoid(self.gate(torch.cat([query_pool,has_relation.float()[:,None]],-1))).squeeze(-1)
        logits=(generic+alpha[:,None]*structured).masked_fill(~mask,-1e4)
        mf=mask.float()[...,None]; pooled=(x*mf).sum(1)/mf.sum(1).clamp_min(1); best=torch.where(mask.any(1),logits.max(1).values,torch.zeros(x.shape[0],device=x.device)); null_logit=self.null(torch.cat([pooled,query_pool,best[:,None]],-1)).squeeze(-1)
        return {'candidate_logits':logits,'null_logit':null_logit,'target_entity_logits':target_score,'generic_logits':generic,'structured_logits':structured,'gate':alpha}

model=FinalDODCore(576,TEXT_DIM).to(DEV)
print('trainable parameters M',sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6)


## 7. Training

Only the DOD core receives gradients. The detector, RegionEncoder, and DistilBERT are frozen. Loss combines candidate focal BCE, explicit null BCE, and target-entity auxiliary focal BCE.

In [ ]:
def move_batch(B):
    for k in ['boxes','scores','features','label_ids','mask','y','null','rel','has_rel']: B[k]=B[k].to(DEV,non_blocking=True)
    return B
@torch.inference_mode()
def language_for_batch(B):
    tt,tm,qp=encode_text(B['queries']); _,_,tp=encode_text(B['targets'],24); _,_,ap=encode_text(B['anchors'],24); return tt,tm,qp,tp,ap
def forward_batch(B):
    B=move_batch(B); tt,tm,qp,tp,ap=language_for_batch(B); le=LABEL_EMB.to(DEV)[B['label_ids']]
    return model(B['features'],B['boxes'],B['scores'],le,B['mask'],tt,tm,qp,tp,ap,B['rel'],B['has_rel']),B
def focal_bce(logits,y,mask,gamma=2.,alpha=.75):
    z=logits[mask]; t=y[mask]
    if z.numel()==0: return logits.sum()*0
    b=F.binary_cross_entropy_with_logits(z,t,reduction='none'); p=torch.sigmoid(z); pt=torch.where(t>0,p,1-p); a=torch.where(t>0,torch.full_like(t,alpha),torch.full_like(t,1-alpha)); return (a*(1-pt).pow(gamma)*b).mean()
def compute_loss(O,B):
    lc=focal_bce(O['candidate_logits'],B['y'],B['mask']); ln=F.binary_cross_entropy_with_logits(O['null_logit'],B['null']); le=focal_bce(O['target_entity_logits'],B['y'],B['mask']); return cfg.w_candidate*lc+cfg.w_null*ln+cfg.w_entity*le,{'candidate':float(lc.detach()),'null':float(ln.detach()),'entity':float(le.detach())}

optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay); scaler=torch.amp.GradScaler('cuda',enabled=DEV.type=='cuda'); best=float('inf'); bad=0; history=[]
for epoch in range(1,cfg.epochs+1):
    model.train(); train_losses=[]; injected=0; samples=0
    for B in tqdm(train_loader,desc=f'train {epoch}'):
        injected+=sum(m['injected'] for m in B['meta']); samples+=len(B['meta']); optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEV.type,dtype=torch.float16,enabled=DEV.type=='cuda'):
            O,B=forward_batch(B); loss,parts=compute_loss(O,B)
        if not torch.isfinite(loss): raise FloatingPointError(f'non-finite loss epoch={epoch}: {loss}')
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); scaler.step(optimizer); scaler.update(); train_losses.append(float(loss.detach()))
    model.eval(); val_losses=[]
    with torch.inference_mode():
        for B in tqdm(val_loader,desc=f'val {epoch}'):
            O,B=forward_batch(B); loss,_=compute_loss(O,B); val_losses.append(float(loss))
    tr=float(np.mean(train_losses)); va=float(np.mean(val_losses)); rec={'epoch':epoch,'train_loss':tr,'val_loss':va,'gt_injected_per_sample':injected/max(samples,1)}; history.append(rec); print(rec)
    if va<best-1e-5:
        best=va; bad=0; torch.save(model.state_dict(),CK/'best_dod_core.pt')
    else:
        bad+=1
        if bad>=cfg.patience: print('early stop'); break
pd.DataFrame(history).to_csv(ROOT/'training_history.csv',index=False)
model.load_state_dict(torch.load(CK/'best_dod_core.pt',map_location=DEV,weights_only=True)); model.eval()


## 8. Validation calibration + held-out evaluation

First fit separate scalar temperatures for candidate and null logits on validation. Then grid-search candidate/null thresholds. Report object matching at IoU 0.50 and 0.75 plus no-target behavior.

In [ ]:
@torch.inference_mode()
def collect(loader):
    records=[]
    for B0 in tqdm(loader,desc='collect logits'):
        meta=B0['meta']; O,B=forward_batch(B0)
        for i,m in enumerate(meta):
            records.append({'sample_id':m['sample_id'],'image_id':m['image_id'],'file_name':m['file_name'],'gt':m['gt'].cpu(),'boxes':B['boxes'][i].cpu(),'mask':B['mask'][i].cpu(),'y':B['y'][i].cpu(),'logits':O['candidate_logits'][i].float().cpu(),'null_label':float(B['null'][i].cpu()),'null_logit':float(O['null_logit'][i].float().cpu())})
    return records
val_records=collect(val_loader)

def fit_temperature(logits,labels):
    logits=logits.float().to(DEV); labels=labels.float().to(DEV); logT=torch.zeros(1,device=DEV,requires_grad=True); opt=torch.optim.LBFGS([logT],lr=.1,max_iter=50)
    def closure():
        opt.zero_grad(); T=logT.exp().clamp(.05,20); L=F.binary_cross_entropy_with_logits(logits/T,labels); L.backward(); return L
    opt.step(closure); return float(logT.detach().exp().clamp(.05,20).cpu())
cl=torch.cat([r['logits'][r['mask']] for r in val_records]); cy=torch.cat([r['y'][r['mask']] for r in val_records]); nl=torch.tensor([r['null_logit'] for r in val_records]); ny=torch.tensor([r['null_label'] for r in val_records])
CAND_T=fit_temperature(cl,cy); NULL_T=fit_temperature(nl,ny); print('temperatures',CAND_T,NULL_T)

def decode_record(r,ct,nt):
    p=torch.sigmoid(r['logits'][r['mask']]/CAND_T); b=r['boxes'][r['mask']]; nullp=float(torch.sigmoid(torch.tensor(r['null_logit']/NULL_T)))
    if len(p)==0 or nullp>=nt or float(p.max())<ct: return torch.zeros((0,4)),torch.zeros(0)
    k=torch.where(p>=ct)[0]; b=b[k]; p=p[k]; keep=nms(b,p,cfg.nms_iou)[:cfg.max_outputs]; return b[keep],p[keep]
def evaluate_records(records,ct,nt,iou_thr=.5):
    TP=FP=FN=0; no_n=no_ok=0
    for r in records:
        pb,_=decode_record(r,ct,nt); gt=r['gt']; M=box_iou(pb,gt); used=set(); tp=0
        for i in range(len(pb)):
            if len(gt):
                order=torch.argsort(M[i],descending=True).tolist()
                for j in order:
                    if float(M[i,j])<iou_thr: break
                    if j not in used: used.add(j); tp+=1; break
        TP+=tp; FP+=len(pb)-tp; FN+=len(gt)-tp
        if len(gt)==0: no_n+=1; no_ok+=int(len(pb)==0)
    precision=TP/max(TP+FP,1); recall=TP/max(TP+FN,1); f1=2*precision*recall/max(precision+recall,1e-9)
    return {'precision':precision,'recall':recall,'f1':f1,'no_target_accuracy':no_ok/max(no_n,1),'TP':TP,'FP':FP,'FN':FN,'no_target_n':no_n}
best_cal=None
for ct in np.arange(.20,.71,.05):
    for nt in np.arange(.30,.81,.05):
        m=evaluate_records(val_records,float(ct),float(nt),.5); objective=.70*m['f1']+.30*m['no_target_accuracy']
        if best_cal is None or objective>best_cal[0]: best_cal=(objective,float(ct),float(nt),m)
_,CAND_THR,NULL_THR,val_metrics=best_cal; print('thresholds',CAND_THR,NULL_THR,'val',val_metrics)
test_records=collect(test_loader)
test_metrics={'iou_0.50':evaluate_records(test_records,CAND_THR,NULL_THR,.50),'iou_0.75':evaluate_records(test_records,CAND_THR,NULL_THR,.75)}
print(json.dumps(test_metrics,indent=2))


## 9. Package the final DOD artifact

The bundle stores learned A0+A1/null weights, architecture/config, relation vocabulary, label vocabulary, and calibration. The frozen RegionEncoder and text encoder are saved beside it.

In [ ]:
ART.mkdir(parents=True,exist_ok=True)
bundle={'version':'dod-v3-a0-a1','model_state':{k:v.detach().cpu() for k,v in model.state_dict().items()},'config':asdict(cfg),'text_dim':TEXT_DIM,'relation_to_id':REL2ID,'label_to_id':LABEL2ID,'label_embeddings':LABEL_EMB.half(),'calibration':{'candidate_temperature':CAND_T,'null_temperature':NULL_T,'candidate_threshold':CAND_THR,'null_threshold':NULL_THR},'proposal_model':cfg.proposal_model}
torch.save(bundle,ART/'final_dod_bundle.pt'); shutil.copy2(ROOT/'region_encoder.pt',ART/'region_encoder.pt')
if (ROOT/'proposal_model.pt').exists(): shutil.copy2(ROOT/'proposal_model.pt',ART/'proposal_model.pt')
text_encoder.cpu().save_pretrained(ART/'text_encoder',safe_serialization=True); tokenizer.save_pretrained(ART/'text_encoder')
json.dump({'validation':val_metrics,'test':test_metrics},open(ART/'metrics.json','w'),indent=2); json.dump(asdict(cfg),open(ART/'config.json','w'),indent=2); shutil.copy2(ROOT/'training_history.csv',ART/'training_history.csv')
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for ch in iter(lambda:f.read(1024*1024),b''): h.update(ch)
    return h.hexdigest()
json.dump({'dataset_repo':cfg.dataset_repo,'train_samples':len(train_df),'val_samples':len(val_df),'test_samples':len(test_df),'cache_signature':cache_signature,'grefs_sha256':sha256(ANN/'grefs(unc).json'),'instances_sha256':sha256(ANN/'instances.json')},open(ART/'dataset_metadata.json','w'),indent=2)
print('FINAL MODEL',ART/'final_dod_bundle.pt'); print([p.name for p in ART.iterdir()])


## 10. Reload the packaged model and run real inference

This cell deliberately discards training objects, loads only the artifact directory, and executes the final architecture on a held-out image. If this cell succeeds, the produced bundle is self-consistent.

In [ ]:
del model,text_encoder; gc.collect();
if torch.cuda.is_available(): torch.cuda.empty_cache()

class FinalDODRuntime:
    def __init__(self,artifact,device=None):
        from ultralytics import YOLOE
        self.dir=Path(artifact); self.device=torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu')); B=torch.load(self.dir/'final_dod_bundle.pt',map_location='cpu',weights_only=False); self.B=B; self.cfg=Config(**B['config']); self.rel2id=B['relation_to_id']; self.label2id=B['label_to_id']; self.label_emb=B['label_embeddings'].float().to(self.device); self.cal=B['calibration']
        self.tok=AutoTokenizer.from_pretrained(self.dir/'text_encoder'); self.txt=AutoModel.from_pretrained(self.dir/'text_encoder').to(self.device).eval(); self.region=RegionEncoder(False).to(self.device).eval(); self.region.load_state_dict(torch.load(self.dir/'region_encoder.pt',map_location=self.device,weights_only=True)); self.core=FinalDODCore(576,B['text_dim']).to(self.device).eval(); self.core.load_state_dict(B['model_state'])
        p=self.dir/'proposal_model.pt'; self.proposal=YOLOE(str(p) if p.exists() else B['proposal_model'])
    @torch.inference_mode()
    def enc(self,texts,max_len=None):
        t=self.tok(list(texts),padding=True,truncation=True,max_length=max_len or self.cfg.max_text_len,return_tensors='pt'); t={k:v.to(self.device) for k,v in t.items()}; z=self.txt(**t).last_hidden_state; m=t['attention_mask'].bool(); p=(z*m[...,None]).sum(1)/m.sum(1,keepdim=True).clamp_min(1); return z,m,p
    @torch.inference_mode()
    def predict(self,image,query):
        pil=Image.open(image).convert('RGB') if not isinstance(image,Image.Image) else image.convert('RGB'); w,h=pil.size; z=self.proposal.predict(source=np.asarray(pil),imgsz=self.cfg.proposal_imgsz,conf=self.cfg.proposal_conf,max_det=self.cfg.proposal_max_det,device=0 if self.device.type=='cuda' else 'cpu',half=self.device.type=='cuda',verbose=False)[0]
        if z.boxes is None or len(z.boxes)==0: return []
        b=z.boxes.xyxy.detach().float().cpu(); s=z.boxes.conf.detach().float().cpu(); ids=z.boxes.cls.detach().long().cpu(); order=torch.argsort(s,descending=True)[:self.cfg.top_k]; b=b[order]; s=s[order]; labels=[str(z.names[int(ids[i])]) for i in order.tolist()]; feat=self.region.encode(pil,b).float().to(self.device); n=len(b); K=self.cfg.top_k
        bn=(b/torch.tensor([w,h,w,h],dtype=torch.float32)).to(self.device); sc=s.to(self.device); lid=torch.tensor([self.label2id.get(x,0) for x in labels],device=self.device); le=self.label_emb[lid]
        def pad(x,shape): return torch.cat([x,torch.zeros(shape,device=self.device,dtype=x.dtype)],0) if len(x)<K else x[:K]
        bn=pad(bn,(K-n,4))[None]; feat=pad(feat,(K-n,576))[None]; sc=pad(sc,(K-n,))[None]; le=pad(le,(K-n,self.B['text_dim']))[None]; mask=torch.zeros((1,K),dtype=torch.bool,device=self.device); mask[0,:n]=True
        target,anchor,rel,has_rel=parse_query(query); tt,tm,qp=self.enc([query]); _,_,tp=self.enc([target],24); _,_,ap=self.enc([anchor],24); O=self.core(feat,bn,sc,le,mask,tt,tm,qp,tp,ap,torch.tensor([self.rel2id[rel]],device=self.device),torch.tensor([has_rel],device=self.device))
        cp=torch.sigmoid(O['candidate_logits'][0,:n]/self.cal['candidate_temperature']); np_=float(torch.sigmoid(O['null_logit'][0]/self.cal['null_temperature']))
        if np_>=self.cal['null_threshold'] or float(cp.max())<self.cal['candidate_threshold']: return []
        keep=torch.where(cp>=self.cal['candidate_threshold'])[0]; bb=b[keep.cpu()]; pp=cp[keep]; kk=nms(bb.to(self.device),pp,self.cfg.nms_iou)[:self.cfg.max_outputs].cpu(); return [{'box':bb[i].tolist(),'score':float(pp[i]),'proposal_label':labels[int(keep[i])]} for i in kk]

runtime=FinalDODRuntime(ART)
example=test_df.iloc[0]; pred=runtime.predict(IMG/example.file_name,example.query); print('query:',example.query); print('GT:',example.gt_boxes); print('prediction:',pred)
im=Image.open(IMG/example.file_name).convert('RGB'); dr=ImageDraw.Draw(im)
for p in pred: dr.rectangle(p['box'],outline='red',width=4)
plt.figure(figsize=(12,8)); plt.imshow(im); plt.axis('off'); plt.title(example.query); plt.show()


In [ ]:
# Optional static prompt-free YOLOE TensorRT export after the learned model is complete.
if cfg.export_yoloe_tensorrt:
    from ultralytics import YOLOE
    exp=YOLOE(str(ART/'proposal_model.pt') if (ART/'proposal_model.pt').exists() else cfg.proposal_model)
    engine=exp.export(format='engine',imgsz=cfg.proposal_imgsz,half=True,dynamic=False)
    print('TensorRT proposal engine:',engine)
else:
    print('TensorRT export disabled. Final learned DOD model:',ART/'final_dod_bundle.pt')


## Final output

A successful full Run All produces `../artifacts/dod_final_model/final_dod_bundle.pt` plus the frozen visual/text components, calibration, metrics, training history, and dataset metadata. `FinalDODRuntime` above is the reference end-to-end inference implementation.